# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an end-to-end workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by the Croissant schema at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata:
print(f"Dataset: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Published: {dataset.metadata.datePublished}")
print('\nDescription:')
print(dataset.metadata.description)

## 2. Data Overview
Let's list all available Record Sets, their `@id`, and summarize the available fields within each. We'll use the proper Croissant API to reference entities by `@id`.

In [ ]:
# List all Record Sets and their fields by @id
record_sets = dataset.metadata.recordSet
print(f"Number of record sets: {len(record_sets)}\n")
for r in record_sets:
    print(f"Record Set: {r['@id']}")
    if 'field' in r:
        fields = r['field']
        print(f"  Fields ({len(fields)}):")
        for f in fields:
            if isinstance(f, dict):
                print(f"    - {f['@id']} : {f.get('name','')} (type: {f.get('dataType','')})")
    print()

## 3. Data Extraction
We'll load data for each record set separately. All entities (record sets, fields, columns) will be referenced by their `@id` as specified by the schema. 

**Note**: The true `@id` values must be extracted from the record sets above. For demonstration, we load all record sets into a Python dictionary, referencing by their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [r['@id'] for r in dataset.metadata.recordSet]
dfs = {}
for record_set_id in record_set_ids:
    # Use .records to fetch all records for the given record_set @id
    rows = list(dataset.records(record_set=record_set_id))
    if len(rows) > 0:
        dfs[record_set_id] = pd.DataFrame(rows)
    else:
        dfs[record_set_id] = pd.DataFrame()  # Empty if no records
    print(f"Loaded {len(rows)} records for Record Set: {record_set_id}")

# Display columns for each DataFrame
for record_set_id, df in dfs.items():
    print(f"\nFirst 5 columns for '{record_set_id}': {df.columns[:5].tolist() if not df.empty else '(empty)'}")
    if not df.empty:
        display(df.head(2))

# For demonstration, choose the first non-empty record set for further analysis:
main_record_set_id = next((k for k, df in dfs.items() if not df.empty), None)
main_df = dfs[main_record_set_id] if main_record_set_id else None
print(f"\nMain record set selected for analysis: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
We'll apply some typical processing steps:
- Filter records on a numeric field (e.g., age or interval).
- Normalize a numeric column.
- Group by a categorical column (e.g., cancer type, sex, MSI status).

All fields/columns are referenced by their schema `@id` (or matching DataFrame column names if the `@id` is present as such).

In [ ]:
# Let's pick a likely numeric field.
# Run this cell to display all column names for discovery:
if main_df is not None:
    print("DataFrame columns:")
    print(main_df.columns.tolist())
else:
    print("No main DataFrame loaded.")

In [ ]:
# Suppose the field representing 'Interval_between_cancers' exists and has @id 'interval_between_cancers' in the schema.
# Replace field IDs below based on schema from previous outputs if different.
numeric_field_id = 'interval_between_cancers'  # Example field @id; update if necessary

# Set threshold for filtering
threshold = 12

if main_df is not None and numeric_field_id in main_df.columns:
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records in '{main_record_set_id}' with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' values:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a categorical variable, e.g., MSI status or cancer_type
    # Suppose 'msi_status' is a field @id
    group_field_id = 'msi_status'  # Replace with actual field @id
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df)
else:
    print(f"Column '{numeric_field_id}' not found in main DataFrame.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and relationships with a categorical variable, referencing all data fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot for numeric_field_id by a categorical group_field_id (e.g., 'msi_status')
    if 'msi_status' in main_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=main_df, x='msi_status', y=numeric_field_id)
        plt.title(f"{numeric_field_id} by MSI Status")
        plt.xlabel('msi_status (@id)')
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Required numeric field is missing for visualization.")

## 6. Conclusion

- We loaded the FAIR^2 clinical oncology dataset using the `mlcroissant` library, referencing all entities by their `@id` as per the Croissant schema specification.
- We explored record sets, identified fields, and extracted data for analysis.
- Simple EDA was performed: records filtered and normalized by a numeric attribute, grouped by key variables such as MSI status.
- Visualizations showcased numeric field distributions and their relationship to molecular status.

This workflow can be adapted to other Croissant-compliant datasets, ensuring reproducible and schema-based data exploration across research projects.